# GPT-2 Using HuggingFance and Tensorflow
En este Notebook se realizar?n los procesamientos necesarios para hacer fine tuning de GPT-2 para la generaci?n de memes en espa?ol.


## 1. Carga de Informaci?n
Durante la primera fase de creaci?n, se utilizar? ?nicamente uno de los conjuntos de datos disponibles. 

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv('../data/bases_no_tilde/useful_scrapymemes1m_no_tilde.csv')

In [ ]:
df = df[(~df.text_1.isna()) & (~df.text_2.isna())]
df["text_1"] = df["text_1"].str.lower()
df["text_2"] = df["text_2"].str.lower()
df['meme_id'] = df['meme_id'].apply('{:0>4}'.format)

In [ ]:
df.head()

In [ ]:
def GPT_text(row):
    meme_id = row.meme_id
    text_1 = row.text_1 
    text_2 = row.text_2 
    return '<|'+meme_id+'|>' + text_1.strip() + ' | ' + text_2.strip()

df['GPT_text'] = df.apply(GPT_text, axis = 1)

In [ ]:
df[['GPT_text', 'meme_id', 'text_1', 'text_2']].head()

In [ ]:
train_text, test_text, train_id, test_id  = train_test_split(df['GPT_text'], df['meme_id'], train_size=.9, random_state=42) # Para hacerlo estratificado

In [ ]:
print(f'Memes en Train: {len(train_text)}\nMemes en Test: {len(test_text)}')

In [ ]:
model_data_path = '../model_data'

train = os.path.join(model_data_path, 'train_trial.txt')
test = os.path.join(model_data_path, 'test_trial.txt')

with open(train, 'w') as file_handle:
  file_handle.write("<|endoftext|>".join(train_text.values.tolist()))

with open(test, 'w') as file_handle:
  file_handle.write("<|endoftext|>".join(test_text.values.tolist()))

# Entrenamiento de Modelo

In [ ]:
from transformers import TFGPT2LMHeadModel
from transformers import GPT2Tokenizer


tokenizer = GPT2Tokenizer.from_pretrained("datificate/gpt2-small-spanish", from_pt = True)
model = TFGPT2LMHeadModel.from_pretrained("datificate/gpt2-small-spanish")

In [ ]:
tokenizer.encode("<0004>hace una chilena en el poli | se desnuca<|endoftext|>", return_tensors='tf')